# SSAFY 16기 2회차 AI 챌린지 — 텍스트 이미지 VQA 솔루션 (v4)

베이스라인(v3) 대비 **점수를 올리는 데 실제로 효과가 큰 순서대로** 개선한 버전입니다.
코딩을 잘 모르셔도 **위에서부터 셀을 차례대로 실행**하면 `submission.csv`가 만들어집니다.

---

## 무엇을 바꿨고, 왜 점수가 오르는가

| # | 개선 항목 | 베이스라인(v3) | 이 노트북(v4) | 왜 중요한가 |
|---|---|---|---|---|
| 1 | **이미지 해상도** | `max_pixels = 256×28×28` (≈ 448×448) | 학습 512~1024, 추론 1024~1280 `×28×28` | 이 대회는 **이미지 속 글자를 읽는(OCR) 문제**입니다. 448px로 줄이면 간판·메뉴판 글씨가 뭉개져서 모델이 아예 못 읽습니다. **가장 큰 점수 상승 요인** |
| 2 | **모델 크기** | Qwen2.5-VL-**3B** | Qwen2.5-VL-**7B** (GPU 자동 감지, 부족하면 3B) | 7B가 한국어 OCR·추론에서 3B보다 확연히 강함 |
| 3 | **정답 뽑는 방식** | 텍스트를 생성한 뒤 문자열 파싱 | **로짓(확률) 스코어링** | 생성 실패 시 baseline은 무조건 `"a"`로 찍습니다. v4는 a/b/c/d 네 토큰의 **확률을 직접 비교**해서 파싱 실패가 **0건** |
| 4 | **보기 순서 TTA** | 없음 | **4가지 순환 치환 평균** | 모델은 "(a)를 고르는 버릇" 같은 **위치 편향**이 있습니다. 보기 순서를 4번 돌려가며 평균내면 편향이 사라집니다 |
| 5 | **학습 라벨 마스킹** | `labels = input_ids` (질문·이미지까지 전부 학습) | **정답 글자만 학습** | v3는 "질문을 외우는" 학습을 합니다. 정답 예측력이 거의 안 오르고 과적합만 커집니다. **버그성 개선** |
| 6 | **학습 데이터량** | 1,000개만 사용 | **전체 사용** (+ 보기 순서 셔플 증강) | 데이터를 버릴 이유가 없음 |
| 7 | **과적합 방지** | val loss만 출력 | **val 정확도로 best 체크포인트 선택 + 얼리스톱 + 제로샷과 비교해 더 좋은 쪽 자동 채택** | 파인튜닝이 오히려 나빠지면 **자동으로 제로샷을 씁니다** |
| 8 | **EXIF 회전 보정** | 없음 | `ImageOps.exif_transpose` | 휴대폰 사진은 90° 돌아가 저장된 게 많습니다. 글자가 누워 있으면 OCR 실패 |
| 9 | **dev 데이터 활용** | 미사용 | 교육생 응답 5개의 **다수결 의사라벨**(4/5 이상 합의만) | 규정상 dev 증강 허용. 학습 데이터 추가 확보 |
| 10 | **정답 분포 보정** | 없음 | val에서 **검증된 경우에만** 사전확률 보정 적용 | 효과 없으면 자동으로 `alpha=0`(미적용) |

> **핵심 한 줄**: *해상도를 올리고, 생성 대신 확률을 비교하고, 보기 순서를 섞어 평균내는 것* 세 가지만으로도 베이스라인보다 큰 폭으로 올라갑니다. 파인튜닝은 그 위에 얹는 보너스이며, **검증 성적이 나쁘면 자동으로 버립니다.**

---

## 모델 버전업 경로

이 노트북은 **모델을 한 줄로 갈아끼울 수 있게** 설계되어 있습니다 (`CFG["model_key"]`).
`AutoModelForImageTextToText`로 로드하고, 해상도는 토큰 예산으로 다루며,
assistant 표식은 chat template에서 자동으로 뽑아내기 때문입니다.

| 단계 | 모델 | 필요 VRAM(4bit) | 기대 효과 |
|---|---|---|---|
| **지금** | `qwen2.5-vl-7b` | ~6GB | 기준선 |
| **1순위** | **`qwen3-vl-8b`** | ~7GB | OCR 지원 언어 10→39개, **흐림·기울어짐·저조도에 강함**, DeepStack으로 미세 디테일 강화 — 이 대회 데이터(휴대폰으로 찍은 간판)에 정확히 들어맞음 |
| 2순위 | `qwen3-vl-32b` | ~20GB | 확실히 더 강하지만 다운로드·추론 비용 큼 |
| 대안 | `qwen3-vl-30b-a3b` | ~18GB | MoE(활성 3B)라 **추론이 빠름**. 단 VRAM은 30B분 필요 |
| 저사양 | `qwen3-vl-4b` | ~4GB | 3B 대비 업그레이드 |

기본값은 `qwen3-vl-8b`이고, `transformers`가 4.57 미만이면 **자동으로 Qwen2.5-VL로 내려갑니다.**

> **주의:** "한국어 데이터니까 한국어 특화 모델"은 이 경우 답이 아닐 수 있습니다.
> 한국어 벤치마크(KorMedMCQA-V)에서 한국어 특화 최고 모델(VARCO-VISION-2.0-14B)이 43.2%인 반면
> Qwen3-VL-32B-Thinking은 83.7%였습니다. **반드시 본인 검증 세트로 직접 비교하세요** —
> 이 노트북의 후보 비교 장치(11번 셀)가 그 용도입니다.

---
# 0. 환경 준비

아래 셀을 실행한 뒤 **런타임 → 세션 다시 시작**(Colab) 을 한 번 해주세요.
그 다음 이 셀은 건너뛰고 1번부터 이어서 실행하면 됩니다.

> 베이스라인은 `transformers`를 GitHub 최신 소스에서 설치했는데, 그 방식은 날짜에 따라 갑자기
> 깨지는 경우가 있습니다. 여기서는 **동작이 검증된 릴리스 버전 범위**를 설치합니다.

In [ ]:
# ⚠️ 이 셀 실행 후 "런타임 → 세션 다시 시작"을 한 번 해주세요.
import sys, subprocess

PKGS = [
    "transformers>=4.57.0",   # Qwen3-VL은 4.57.0 이상 필요
    "accelerate>=0.34.0",
    "peft>=0.13.2",
    "bitsandbytes>=0.43.0",
    "datasets",
    "pillow",
    "pandas",
    "tqdm",
    "scikit-learn",
    "matplotlib",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS], check=False)
print("설치 완료 → 런타임 다시 시작(Restart) 후 1번 셀부터 실행하세요.")

In [ ]:
# 설치 확인 (세션 다시 시작 후 실행)
import torch, transformers, platform
print("python      :", platform.python_version())
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name}  /  VRAM {p.total_memory/1024**3:.1f} GB")
    print("bf16 지원   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️ GPU가 없습니다. Colab 상단 [런타임 → 런타임 유형 변경 → GPU]를 선택하세요.")

---
# 1. 데이터 준비

`DATA_ROOT` 아래에 아래 구조가 있으면 됩니다.

```
DATA_ROOT/
├── train.csv, test.csv, dev.csv, sample_submission.csv
├── train/  train_0001.jpg ...
├── test/   test_0001.jpg ...
└── dev/    dev_0001.jpg ...
```

Colab에서 구글 드라이브의 zip을 쓰신다면 아래 셀의 `UNZIP_FROM`에 zip 경로를 넣으세요.
이미 압축을 풀었거나 로컬 PC에서 쓰신다면 `UNZIP_FROM = None` 으로 두고 `DATA_ROOT`만 맞춰주세요.

In [ ]:
import os, glob

# ── 여기만 본인 환경에 맞게 수정 ──────────────────────────────
DATA_ROOT   = "/content"                                  # csv와 이미지 폴더가 있는 위치
UNZIP_FROM  = "/content/drive/MyDrive/ssafy-16-2-ai.zip"   # 압축 풀 zip 경로. 필요 없으면 None
OUTPUT_DIR  = "/content/vqa_out"                           # 체크포인트/제출물 저장 위치
# ─────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

def _already_extracted(root):
    return os.path.exists(os.path.join(root, "train.csv")) and os.path.isdir(os.path.join(root, "train"))

if UNZIP_FROM and not _already_extracted(DATA_ROOT):
    # 구글 드라이브가 필요하면 마운트
    if UNZIP_FROM.startswith("/content/drive") and not os.path.exists("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    print("압축 해제 중... (수 분 소요)")
    os.system(f'7z x -y -o"{DATA_ROOT}" "{UNZIP_FROM}" > /dev/null 2>&1 || unzip -q -o "{UNZIP_FROM}" -d "{DATA_ROOT}"')

assert _already_extracted(DATA_ROOT), (
    f"{DATA_ROOT} 에서 train.csv / train 폴더를 찾지 못했습니다. DATA_ROOT 경로를 확인하세요.\n"
    f"현재 내용: {os.listdir(DATA_ROOT)[:20]}"
)
print("데이터 준비 완료:", DATA_ROOT)
for name in ["train.csv", "test.csv", "dev.csv", "sample_submission.csv"]:
    p = os.path.join(DATA_ROOT, name)
    print(f"  {'✅' if os.path.exists(p) else '⬜'} {name}")

In [ ]:
import pandas as pd, numpy as np, os

LETTERS = ["a", "b", "c", "d"]

def _read(name):
    p = os.path.join(DATA_ROOT, name)
    return pd.read_csv(p) if os.path.exists(p) else None

train_df  = _read("train.csv")
test_df   = _read("test.csv")
dev_df    = _read("dev.csv")
sample_sub = _read("sample_submission.csv")

assert train_df is not None and test_df is not None, "train.csv / test.csv 가 필요합니다."

def resolve_path(rel):
    '''CSV의 path 컬럼(예: "train/train_0001.jpg")을 실제 절대경로로 변환'''
    rel = str(rel)
    if os.path.isabs(rel) and os.path.exists(rel):
        return rel
    cand = os.path.join(DATA_ROOT, rel)
    if os.path.exists(cand):
        return cand
    # 폴더 구조가 다를 때를 대비한 파일명 기반 폴백
    base = os.path.basename(rel)
    for sub in ("train", "test", "dev", ""):
        c = os.path.join(DATA_ROOT, sub, base)
        if os.path.exists(c):
            return c
    return cand  # 없으면 그대로 반환(뒤에서 에러로 잡힘)

def prepare(df, has_answer):
    df = df.copy()
    df["abs_path"] = df["path"].map(resolve_path)
    for c in ["question", "a", "b", "c", "d"]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    if has_answer:
        df["answer"] = df["answer"].astype(str).str.strip().str.lower().str[0]
        df = df[df["answer"].isin(LETTERS)].reset_index(drop=True)
    return df.reset_index(drop=True)

train_df = prepare(train_df, True)
test_df  = prepare(test_df,  False)

print(f"train: {len(train_df):,}행   test: {len(test_df):,}행   dev: {0 if dev_df is None else len(dev_df):,}행")

# 이미지 존재 여부 점검
missing = sum(0 if os.path.exists(p) else 1 for p in train_df["abs_path"].head(200))
print("train 이미지 샘플 200개 중 누락:", missing)

# 정답 분포 확인 — 한쪽으로 쏠려 있으면 뒤에서 '사전확률 보정'으로 활용합니다.
dist = train_df["answer"].value_counts(normalize=True).reindex(LETTERS).fillna(0)
print("\n정답 분포:")
for L in LETTERS:
    print(f"  {L}: {dist[L]*100:5.1f} %  {'█'*int(dist[L]*100)}")
ANSWER_PRIOR = dist.values.astype(np.float64)
ANSWER_PRIOR = ANSWER_PRIOR / ANSWER_PRIOR.sum()
print("\n예시 문항:")
print(train_df.iloc[0][["question", "a", "b", "c", "d", "answer"]].to_string())

---
# 2. 설정 (CONFIG)

GPU를 자동으로 감지해서 **모델 크기 / 해상도 / 배치**를 알아서 고릅니다.
값을 직접 바꾸고 싶으면 아래 `CFG` 딕셔너리만 수정하세요.

### 해상도(`*_max_tokens`)가 왜 중요한지
VLM은 이미지를 정사각형 조각으로 잘라 **비전 토큰**으로 바꿔서 봅니다.
`*_max_tokens`는 **한 이미지에 허용할 비전 토큰 수의 상한**입니다.

베이스라인의 `256`은 약 `448×448`픽셀입니다. 간판 글씨를 읽기엔 턱없이 부족합니다.
`1280`이면 약 `1130×1130`까지 살아남습니다. **메모리가 허락하는 한 크게** 가져가세요.

> ⚠️ **토큰 1개가 담는 픽셀 수는 모델마다 다릅니다.**
> - Qwen2.5-VL : `28×28` 픽셀 = 토큰 1개
> - Qwen3-VL : `32×32` 픽셀 = 토큰 1개
>
> 그래서 이 노트북은 해상도를 픽셀이 아니라 **토큰 예산**으로 지정하고,
> 픽셀 환산은 `MODEL_ZOO`의 `px_unit`으로 자동 처리합니다.
> 이 부분을 하드코딩하면 **모델만 바꿨을 때 해상도가 조용히 틀어집니다.**

In [ ]:
import re, torch, transformers

# ══════════════════════════════════════════════════════════════
# 모델 카탈로그 — 여기 있는 키를 CFG["model_key"]에 넣으면 그대로 교체됩니다.
#
# px_unit = "비전 토큰 1개가 담는 픽셀의 한 변 길이"
#   Qwen2.5-VL : patch 14 + 2x2 merge -> 28x28 픽셀 = 토큰 1개
#   Qwen3-VL   : patch 16 + 2x2 merge -> 32x32 픽셀 = 토큰 1개
#   ★ 모델마다 다릅니다. 같은 "토큰 예산"이라도 Qwen3가 약 31% 더 많은 픽셀을 봅니다.
#     이 값을 안 바꾸고 모델만 갈아끼우면 해상도가 조용히 틀어집니다.
# ══════════════════════════════════════════════════════════════
MODEL_ZOO = {
    # key                 HF 저장소                              px_unit  최소 transformers
    "qwen2.5-vl-3b":   dict(repo="Qwen/Qwen2.5-VL-3B-Instruct",     px_unit=28, min_tf="4.51.0"),
    "qwen2.5-vl-7b":   dict(repo="Qwen/Qwen2.5-VL-7B-Instruct",     px_unit=28, min_tf="4.51.0"),
    "qwen2.5-vl-32b":  dict(repo="Qwen/Qwen2.5-VL-32B-Instruct",    px_unit=28, min_tf="4.51.0"),
    "qwen3-vl-4b":     dict(repo="Qwen/Qwen3-VL-4B-Instruct",       px_unit=32, min_tf="4.57.0"),
    "qwen3-vl-8b":     dict(repo="Qwen/Qwen3-VL-8B-Instruct",       px_unit=32, min_tf="4.57.0"),
    "qwen3-vl-32b":    dict(repo="Qwen/Qwen3-VL-32B-Instruct",      px_unit=32, min_tf="4.57.0"),
    "qwen3-vl-30b-a3b":dict(repo="Qwen/Qwen3-VL-30B-A3B-Instruct",  px_unit=32, min_tf="4.57.0"),
}


def _detect_preset():
    if not torch.cuda.is_available():
        return "cpu"
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram >= 38:  return "big"     # A100 40/80G, H100
    if vram >= 21:  return "mid"     # L4 24G, A10 24G, 3090/4090
    if vram >= 14:  return "small"   # T4 16G, 5060Ti 16G, V100 16G
    return "tiny"                    # 12G 이하

PRESET = _detect_preset()

# *_max_tokens = "비전 토큰 예산". 실제 픽셀 = tokens x px_unit^2 (모델별 자동 환산)
PRESETS = {
    "big":   dict(model_key="qwen3-vl-8b", fallback_key="qwen2.5-vl-7b", load_4bit=False,
                  train_max_tokens=1024, infer_max_tokens=1600, train_bs=2, infer_bs=4, max_train=None, epochs=2),
    "mid":   dict(model_key="qwen3-vl-8b", fallback_key="qwen2.5-vl-7b", load_4bit=True,
                  train_max_tokens=768,  infer_max_tokens=1280, train_bs=1, infer_bs=2, max_train=None, epochs=1),
    "small": dict(model_key="qwen3-vl-8b", fallback_key="qwen2.5-vl-7b", load_4bit=True,
                  train_max_tokens=512,  infer_max_tokens=1024, train_bs=1, infer_bs=1, max_train=4000, epochs=1),
    "tiny":  dict(model_key="qwen3-vl-4b", fallback_key="qwen2.5-vl-3b", load_4bit=True,
                  train_max_tokens=512,  infer_max_tokens=1024, train_bs=1, infer_bs=1, max_train=3000, epochs=1),
    "cpu":   dict(model_key="qwen3-vl-4b", fallback_key="qwen2.5-vl-3b", load_4bit=False,
                  train_max_tokens=256,  infer_max_tokens=512,  train_bs=1, infer_bs=1, max_train=50,   epochs=1),
}

CFG = dict(PRESETS[PRESET])

# ── 공통 설정 (필요하면 직접 수정) ────────────────────────────
CFG.update(
    seed            = 42,
    min_tokens      = 64,      # 최소 비전 토큰 수
    max_image_side  = 2048,    # PIL 디코딩 후 사전 축소 (속도용, 품질 영향 없음)

    # 파인튜닝
    run_finetune    = True,    # False면 제로샷만으로 제출물 생성 (빠르고 이미 강력함)
    lora_r          = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.10,
    lr              = 1e-4,
    weight_decay    = 0.01,
    grad_accum      = 8,
    max_grad_norm   = 1.0,
    warmup_ratio    = 0.03,
    evals_per_epoch = 4,       # 에폭당 검증 횟수 (best 체크포인트 선택용)
    patience        = 2,       # 연속 N회 개선 없으면 조기 종료
    num_workers     = 2,

    # 검증 세트
    val_ratio       = 0.10,
    val_min         = 300,
    val_max         = 1000,
    val_eval_size   = 400,     # 최종 비교(TTA 포함)에 쓸 검증 샘플 수 — 시간 절약용

    # 추론 TTA
    n_perm          = 4,       # 보기 순서 순환 치환 횟수 (1=끔, 4=최대). 4 권장
    train_n_perm    = 1,       # 학습 중 빠른 검증용

    # dev 의사라벨
    use_dev_pseudo  = True,
    dev_min_agree   = 4,       # 교육생 5명 응답 중 최소 4명 일치한 문항만 사용
)

# 모델을 직접 지정하고 싶으면 아래 주석을 풀고 MODEL_ZOO의 키를 넣으세요.
# CFG["model_key"] = "qwen3-vl-32b"       # VRAM 24GB+ & 4bit 권장
# CFG["model_key"] = "qwen3-vl-30b-a3b"   # MoE: 총 30B / 활성 3B — 빠르지만 VRAM은 30B분 필요
# ─────────────────────────────────────────────────────────────

# ── transformers 버전을 확인해 감당 가능한 모델로 자동 조정 ──
def _ver(s):
    return tuple(int(x) for x in re.findall(r"\d+", s)[:3])

_tf_now = _ver(transformers.__version__)
_key = CFG["model_key"]
if _ver(MODEL_ZOO[_key]["min_tf"]) > _tf_now:
    _fb = CFG["fallback_key"]
    print(f"⚠️ transformers {transformers.__version__} 은 '{_key}' 를 지원하지 않습니다 "
          f"(>= {MODEL_ZOO[_key]['min_tf']} 필요).")
    print(f"   → '{_fb}' 로 자동 전환합니다. 최신 모델을 쓰려면 0번 셀을 다시 실행하세요.")
    _key = _fb

MODEL_KEY = _key
MODEL_ID  = MODEL_ZOO[MODEL_KEY]["repo"]
PX_UNIT   = MODEL_ZOO[MODEL_KEY]["px_unit"]     # ★ 해상도 환산의 기준
SEED      = CFG["seed"]

import random, numpy as np
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def tokens_to_px(n_tokens):
    return int(n_tokens) * PX_UNIT * PX_UNIT

print(f"감지된 프리셋 : {PRESET}")
print(f"모델          : {MODEL_KEY}  ->  {MODEL_ID}")
print(f"              4bit={CFG['load_4bit']}, dtype={str(DTYPE).split('.')[-1]}, 토큰당 {PX_UNIT}x{PX_UNIT}px")
print(f"해상도 예산   : 학습 {CFG['train_max_tokens']}토큰 (~{tokens_to_px(CFG['train_max_tokens'])/1e6:.2f}Mpx, "
      f"약 {int(tokens_to_px(CFG['train_max_tokens'])**0.5)}x{int(tokens_to_px(CFG['train_max_tokens'])**0.5)})")
print(f"              추론 {CFG['infer_max_tokens']}토큰 (~{tokens_to_px(CFG['infer_max_tokens'])/1e6:.2f}Mpx, "
      f"약 {int(tokens_to_px(CFG['infer_max_tokens'])**0.5)}x{int(tokens_to_px(CFG['infer_max_tokens'])**0.5)})")
print(f"배치          : 학습 {CFG['train_bs']} (누적 {CFG['grad_accum']}) / 추론 {CFG['infer_bs']}")
print(f"학습 데이터   : {CFG['max_train'] or '전체'}   에폭 {CFG['epochs']}")
if PRESET in ("small", "tiny"):
    print("\n💡 T4/16GB급입니다. 시간이 오래 걸리면 CFG['run_finetune']=False 로 두고")
    print("   제로샷 + 고해상도 + TTA 만으로 제출해도 베이스라인보다 훨씬 좋습니다.")

---
# 3. 이미지 로더 & 프롬프트

### EXIF 회전 보정이 왜 필요한가
휴대폰으로 찍은 사진은 **실제 픽셀은 가로인데 "세로로 돌려서 보여줘"라는 메타데이터(EXIF)** 만
붙어 있는 경우가 많습니다. `Image.open()`만 하면 글자가 90° 누운 채로 모델에 들어갑니다.
`ImageOps.exif_transpose()` 한 줄로 이걸 바로잡습니다. OCR 문제에서 은근히 큰 차이를 냅니다.

In [ ]:
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None   # 초대형 이미지 경고 해제

def load_image(path, max_side=None):
    max_side = max_side or CFG["max_image_side"]
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)      # ★ 휴대폰 사진 회전 보정
    img = img.convert("RGB")
    w, h = img.size
    if max(w, h) > max_side:                # 너무 크면 미리 축소 (디코딩 속도용)
        s = max_side / max(w, h)
        img = img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)
    return img


SYSTEM_INSTRUCT = (
    "당신은 이미지 속의 글자와 장면을 아주 꼼꼼하게 읽어내는 한국어 시각 질의응답(VQA) 전문가입니다. "
    "간판, 메뉴판, 표지판, 안내문에 적힌 작은 글씨까지 정확히 확인한 뒤 답합니다. "
    "반드시 a, b, c, d 중 소문자 한 글자만 출력합니다."
)

def build_prompt(question, options):
    '''options: 화면에 (a),(b),(c),(d) 순서로 보여줄 보기 4개'''
    lines = [f"질문: {question}", "", "보기:"]
    for L, opt in zip(LETTERS, options):
        lines.append(f"({L}) {opt}")
    lines += [
        "",
        "이미지에 실제로 적혀 있는 글자와 세부 정보를 근거로, 위 보기 중 정답 하나를 고르세요.",
        "출력은 오직 a, b, c, d 중 소문자 한 글자입니다. 설명은 쓰지 마세요.",
        "답:",
    ]
    return "\n".join(lines)

def build_messages(question, options, answer_letter=None):
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": build_prompt(question, options)},
        ]},
    ]
    if answer_letter is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer_letter}]})
    return msgs

# 미리보기
print(build_prompt(train_df.iloc[0]["question"],
                   [train_df.iloc[0][c] for c in LETTERS]))

---
# 4. 모델 & 프로세서 로드

7B 모델은 약 16GB 다운로드입니다 (10~20분). 4bit 양자화로 VRAM은 ~6GB만 씁니다.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig

# 모델 종류에 상관없이 로드되는 범용 클래스. Qwen2.5-VL / Qwen3-VL 모두 이걸로 열립니다.
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:                                  # 아주 구버전 대비
    from transformers import Qwen2_5_VLForConditionalGeneration as AutoVLM


def set_pixels(processor, min_tokens, max_tokens):
    '''해상도 한도를 "비전 토큰 예산"으로 지정. 픽셀 환산은 모델의 PX_UNIT을 따른다.'''
    lo, hi = tokens_to_px(min_tokens), tokens_to_px(max_tokens)
    ip = processor.image_processor
    ip.min_pixels, ip.max_pixels = lo, hi
    if isinstance(getattr(ip, "size", None), dict):
        ip.size = {"shortest_edge": lo, "longest_edge": hi}
    return processor

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
set_pixels(processor, CFG["min_tokens"], CFG["infer_max_tokens"])
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── 모델 로드 (버전 차이에 안전하게) ──────────────────────────
load_kw = dict(device_map={"": 0} if DEVICE == "cuda" else "cpu",
               trust_remote_code=True,
               attn_implementation="sdpa")
if CFG["load_4bit"]:
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
    )

try:      # transformers 신버전은 dtype=, 구버전은 torch_dtype=
    base_model = AutoVLM.from_pretrained(MODEL_ID, dtype=DTYPE, **load_kw)
except TypeError:
    base_model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, **load_kw)

base_model.config.use_cache = True
print("모델 로드 완료:", MODEL_ID, f"({base_model.__class__.__name__})")
print("파라미터 수 :", f"{sum(p.numel() for p in base_model.parameters())/1e9:.2f} B")

---
# 5. ★ 핵심 — 로짓 스코어링 + 보기 순서 TTA

### (1) 생성 대신 확률 비교
베이스라인은 모델이 글자를 **생성**하게 한 뒤 문자열에서 `a~d`를 찾습니다.
모델이 `"정답은"` 같은 걸 뱉으면 파싱이 실패하고, 그럴 때 무조건 `"a"`로 찍습니다.

v4는 `assistant` 차례의 **바로 다음 토큰 확률분포**를 꺼내서
`a`, `b`, `c`, `d` 네 토큰의 확률만 비교합니다.

- 파싱 실패 **0건**
- 항상 4지선다 안에서만 고름
- 생성보다 빠름 (토큰 1개 분량 계산)
- **확신도(확률)** 가 나오므로 뒤에서 앙상블·보정에 쓸 수 있음

### (2) 보기 순서 순환 치환 TTA
언어모델에는 "잘 모르겠으면 (a)를 고른다" 같은 **위치 편향**이 있습니다.
같은 문제를 보기 순서만 바꿔 4번 물어보고 **내용 기준으로 확률을 평균**내면 이 편향이 사라집니다.

```
1회차:  (a)=1번보기 (b)=2번 (c)=3번 (d)=4번
2회차:  (a)=2번보기 (b)=3번 (c)=4번 (d)=1번
3회차:  (a)=3번보기 ...
4회차:  (a)=4번보기 ...
```
각 보기가 a/b/c/d 자리에 **정확히 한 번씩** 오므로 위치 효과가 완전히 상쇄됩니다.
비용은 4배지만, 무료로 얻는 정확도 상승폭이 가장 큰 기법 중 하나입니다.

In [ ]:
import numpy as np, torch
from tqdm.auto import tqdm

# a/b/c/d 를 나타낼 수 있는 토큰 id 묶음 (대문자·공백 변형까지 모두 포함해 확률을 합산)
def _choice_id_groups(tok):
    groups = []
    for L in LETTERS:
        ids = set()
        for v in (L, L.upper(), " " + L, " " + L.upper()):
            enc = tok.encode(v, add_special_tokens=False)
            if len(enc) >= 1:
                ids.add(enc[0])
        groups.append(sorted(ids))
    return groups

CHOICE_ID_GROUPS = _choice_id_groups(tokenizer)
print("선지 토큰 id:", dict(zip(LETTERS, CHOICE_ID_GROUPS)))


def _to_device(batch, model):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            v = v.to(model.device, dtype=DTYPE) if v.dtype.is_floating_point else v.to(model.device)
        out[k] = v
    return out


@torch.no_grad()
def _score_texts(model, images, texts):
    '''texts/images 배치에 대해 [B,4] 선지 확률을 반환'''
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"          # 마지막 위치의 로짓을 쓰려면 왼쪽 패딩
    try:
        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt")
    finally:
        tokenizer.padding_side = old_side
    inputs = _to_device(inputs, model)

    gen = model.generate(
        **inputs, max_new_tokens=1, do_sample=False,
        repetition_penalty=1.0,              # 기본 generation_config의 페널티가 로짓을 왜곡하지 않도록
        output_logits=True, output_scores=True, return_dict_in_generate=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    raw = gen.logits[0] if getattr(gen, "logits", None) else gen.scores[0]
    logprobs = torch.log_softmax(raw.float(), dim=-1)
    per_letter = torch.stack(
        [torch.logsumexp(logprobs[:, ids], dim=-1) for ids in CHOICE_ID_GROUPS], dim=1
    )                                        # [B,4] — 4개 선지만 남긴 뒤 재정규화
    return torch.softmax(per_letter, dim=-1).float().cpu().numpy()


class _InferenceMode:
    '''추론 동안 gradient checkpointing 끄고 kv-cache 켜기. 끝나면 학습 상태로 정확히 원복.'''
    def __init__(self, model): self.m = model
    def __enter__(self):
        self.was_training = self.m.training
        self.was_gc = bool(getattr(self.m, "is_gradient_checkpointing", False))
        self.was_cache = bool(getattr(self.m.config, "use_cache", True))
        self.m.eval()
        if self.was_gc:
            try: self.m.gradient_checkpointing_disable()
            except Exception: pass
        self.m.config.use_cache = True
        return self.m
    def __exit__(self, *a):
        # ★ 학습 중 검증에 쓰일 때 gradient checkpointing을 되살리지 않으면
        #    이후 학습 스텝에서 VRAM이 급증해 OOM이 납니다.
        if self.was_gc:
            try:
                self.m.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs={"use_reentrant": False})
            except Exception:
                self.m.gradient_checkpointing_enable()
        self.m.config.use_cache = self.was_cache
        if self.was_training:
            self.m.train()
        return False


def predict_probs(model, df, n_perm=None, max_tokens=None, desc="추론", disable_adapter=False):
    '''
    각 행에 대해 원래 보기 순서(a,b,c,d) 기준 확률 [N,4] 를 반환.
    n_perm 만큼 보기 순서를 순환 치환하며 평균낸다.
    '''
    n_perm = n_perm or CFG["n_perm"]
    max_tokens = max_tokens or CFG["infer_max_tokens"]
    set_pixels(processor, CFG["min_tokens"], max_tokens)

    perms = [[(k + s) % 4 for k in range(4)] for s in range(n_perm)]   # perm[k] = k번 자리에 놓을 원래 보기 index
    probs = np.zeros((len(df), 4), dtype=np.float64)
    bs = max(1, CFG["infer_bs"])

    ctx = model.disable_adapter() if disable_adapter and hasattr(model, "disable_adapter") else _NullCtx()
    with ctx, _InferenceMode(model), torch.no_grad():
        pend_imgs, pend_txts, pend_meta = [], [], []

        def flush():
            if not pend_txts:
                return
            try:
                p = _score_texts(model, pend_imgs, pend_txts)
            except RuntimeError as e:                           # OOM이면 1문장씩 다시 시도
                if "out of memory" not in str(e).lower():
                    raise
                torch.cuda.empty_cache()
                p = np.concatenate([_score_texts(model, [im], [tx])
                                    for im, tx in zip(pend_imgs, pend_txts)], axis=0)
            for (ri, perm), row in zip(pend_meta, p):
                for k in range(4):
                    probs[ri, perm[k]] += row[k]               # 화면 k번 자리 확률 → 원래 보기로 되돌림
            pend_imgs.clear(); pend_txts.clear(); pend_meta.clear()

        for ri in tqdm(range(len(df)), desc=desc, unit="문항"):
            row = df.iloc[ri]
            img = load_image(row["abs_path"])
            options = [row[c] for c in LETTERS]
            for perm in perms:
                shown = [options[perm[k]] for k in range(4)]
                text = processor.apply_chat_template(
                    build_messages(row["question"], shown),
                    tokenize=False, add_generation_prompt=True,
                )
                pend_imgs.append(img); pend_txts.append(text); pend_meta.append((ri, perm))
                if len(pend_txts) >= bs:
                    flush()
        flush()

    probs /= probs.sum(axis=1, keepdims=True)
    return probs


class _NullCtx:
    def __enter__(self): return None
    def __exit__(self, *a): return False


def probs_to_letters(probs):
    return [LETTERS[i] for i in probs.argmax(axis=1)]

def accuracy(probs, gold_letters):
    pred = probs_to_letters(probs)
    return float(np.mean([p == g for p, g in zip(pred, gold_letters)]))

---
# 6. 검증 세트 분리 & 제로샷(학습 없이) 성능 측정

**여기서 만든 검증 세트(val)는 학습에 절대 쓰지 않습니다.**
나중에 "파인튜닝한 모델 vs 제로샷 모델" 중 어느 쪽이 나은지 **공정하게** 판단하는 심판 역할을 합니다.
이게 과적합을 막는 핵심 장치입니다.

In [ ]:
from sklearn.model_selection import train_test_split

n_val = int(np.clip(len(train_df) * CFG["val_ratio"], CFG["val_min"], CFG["val_max"]))
n_val = min(n_val, max(1, len(train_df) // 5))

tr_idx, va_idx = train_test_split(
    np.arange(len(train_df)), test_size=n_val,
    random_state=SEED, stratify=train_df["answer"].values,   # 정답 분포 유지
)
fit_df = train_df.iloc[tr_idx].reset_index(drop=True)
val_df = train_df.iloc[va_idx].reset_index(drop=True)

# 최종 비교용 검증 부분집합 (시간 절약). 층화추출로 대표성 유지.
if len(val_df) > CFG["val_eval_size"]:
    keep, _ = train_test_split(np.arange(len(val_df)), train_size=CFG["val_eval_size"],
                               random_state=SEED, stratify=val_df["answer"].values)
    val_eval_df = val_df.iloc[np.sort(keep)].reset_index(drop=True)
else:
    val_eval_df = val_df.copy()

# 학습 중 빠른 검증용 (더 작게)
val_quick_df = val_eval_df.iloc[:min(250, len(val_eval_df))].reset_index(drop=True)

print(f"학습용 {len(fit_df):,}  /  검증용 {len(val_df):,} (최종비교 {len(val_eval_df)}, 학습중 {len(val_quick_df)})")

In [ ]:
import time

# ── 속도 측정: 전체 소요 시간을 미리 알려줍니다 ─────────────
_bench = val_eval_df.head(4)
_t0 = time.time()
_p = predict_probs(base_model, _bench, n_perm=CFG["n_perm"], desc="속도 측정")
_sec_per_item = (time.time() - _t0) / len(_bench)

# 파이프라인이 정상 동작하는지 즉시 확인 (문제가 있으면 여기서 바로 멈춥니다)
assert _p.shape == (len(_bench), 4), f"확률 배열 모양 이상: {_p.shape}"
assert np.allclose(_p.sum(axis=1), 1.0), "확률 합이 1이 아닙니다"
assert np.isfinite(_p).all(), "확률에 NaN/Inf 가 있습니다"
print("스모크 테스트 통과 — 예시 확률:", np.round(_p[0], 3), "→", LETTERS[_p[0].argmax()])

def eta(n, n_perm=None):
    f = (n_perm or CFG["n_perm"]) / CFG["n_perm"]
    return _sec_per_item * n * f / 60

print(f"\n문항당 {_sec_per_item:.2f}초 (TTA {CFG['n_perm']}회 포함)")
print(f"예상 소요 — 검증 {len(val_eval_df)}문항: {eta(len(val_eval_df)):.0f}분,  테스트 {len(test_df)}문항: {eta(len(test_df)):.0f}분")
if eta(len(test_df)) > 120:
    print("\n⚠️ 테스트 추론이 2시간을 넘습니다. CFG['n_perm']을 2로 낮추면 절반으로 줄어듭니다.")

In [ ]:
# 학습 전 제로샷 성능 (= 파인튜닝이 넘어야 할 기준선)
t0 = time.time()
zs_val_probs = predict_probs(base_model, val_eval_df, desc="제로샷 검증")
zs_val_acc = accuracy(zs_val_probs, val_eval_df["answer"].tolist())
print(f"\n🎯 제로샷 검증 정확도: {zs_val_acc*100:.2f}%   ({(time.time()-t0)/60:.1f}분)")

# 참고: TTA 없이 1회만 했을 때와 비교 (TTA 효과 확인)
zs_val_probs_1 = predict_probs(base_model, val_eval_df, n_perm=1, desc="제로샷(TTA 없음)")
print(f"   TTA 없을 때      : {accuracy(zs_val_probs_1, val_eval_df['answer'].tolist())*100:.2f}%")
print(f"   TTA {CFG['n_perm']}회 적용 시  : {zs_val_acc*100:.2f}%  ← 순환 치환 평균 효과")

---
# 7. dev 데이터 의사라벨 (선택 — 학습 데이터 추가 확보)

`dev.csv`에는 정답이 없지만 **교육생 5명의 응답**이 들어 있습니다.
5명 중 4명 이상이 같은 답을 골랐다면 그 답은 정답일 확률이 매우 높습니다.
이걸 **의사라벨(pseudo-label)** 로 만들어 학습 데이터에 더합니다.
(대회 규정: *"데이터 증강은 학습 데이터 및 개발용 데이터(dev)에 한해 적용 가능"* → 허용됩니다.)

합의가 약한 문항(3:2 같은)은 **버립니다.** 잘못된 라벨을 넣느니 안 넣는 게 낫습니다.

In [ ]:
import re, difflib
from collections import Counter

def _norm(s):
    return re.sub(r"[\s\W_]+", "", str(s)).lower()

def response_to_letter(resp, options):
    '''교육생 응답 한 건을 a~d 로 변환. 못 하면 None.'''
    if resp is None or (isinstance(resp, float) and np.isnan(resp)):
        return None
    s = str(resp).strip()
    if not s:
        return None
    low = s.lower()
    if low in LETTERS:                                   # "a"
        return low
    m = re.match(r"^\(?([abcd])[\)\.\,:\s]", low)        # "a) 역삼", "b. 선릉"
    if m:
        return m.group(1)
    if low in ("1", "2", "3", "4"):                      # "1"
        return LETTERS[int(low) - 1]
    ns = _norm(s)                                        # 보기 본문과 직접 대조
    nopts = [_norm(o) for o in options]
    if ns in nopts:
        return LETTERS[nopts.index(ns)]
    best, score = None, 0.0
    for i, no in enumerate(nopts):
        r = difflib.SequenceMatcher(None, ns, no).ratio()
        if r > score:
            best, score = i, r
    return LETTERS[best] if score >= 0.85 else None


pseudo_df = None
if CFG["use_dev_pseudo"] and dev_df is not None:
    dd = dev_df.copy()
    resp_cols = [c for c in dd.columns if re.fullmatch(r"(응답|response|answer)\s*[1-9]", str(c).strip(), re.I)]
    if not resp_cols:
        resp_cols = [c for c in dd.columns if re.search(r"응답|response", str(c), re.I)]
    print("발견한 응답 컬럼:", resp_cols)

    if resp_cols:
        dd = prepare(dd, has_answer=False)
        rows, n_drop = [], 0
        for _, r in dd.iterrows():
            options = [r[c] for c in LETTERS]
            votes = [response_to_letter(r.get(c), options) for c in resp_cols]
            votes = [v for v in votes if v]
            if not votes:
                n_drop += 1; continue
            letter, cnt = Counter(votes).most_common(1)[0]
            if cnt >= CFG["dev_min_agree"] and cnt / len(votes) >= 0.8:
                rr = r.copy(); rr["answer"] = letter
                rows.append(rr)
            else:
                n_drop += 1
        if rows:
            pseudo_df = pd.DataFrame(rows).reset_index(drop=True)
            print(f"의사라벨 채택 {len(pseudo_df):,}건 / 버림 {n_drop:,}건 "
                  f"(기준: 5명 중 {CFG['dev_min_agree']}명 이상 합의)")
            d2 = pseudo_df["answer"].value_counts(normalize=True).reindex(LETTERS).fillna(0)
            print("의사라벨 정답 분포:", {L: f"{d2[L]*100:.1f}%" for L in LETTERS})
else:
    print("dev 의사라벨 사용 안 함")

---
# 8. 학습용 Dataset / Collator — ★ 라벨 마스킹

### 베이스라인의 치명적 버그
```python
enc["labels"] = enc["input_ids"].clone()    # ← 전부 학습 대상
```
이러면 모델이 **시스템 프롬프트, 질문, 보기, 이미지 토큰까지 전부 "따라 쓰도록"** 학습됩니다.
- 정답을 고르는 능력은 거의 안 늘고,
- 학습 데이터의 질문 문장을 외워버려 **과적합**이 심해집니다.
- 패딩 토큰까지 학습합니다.

### v4의 처리
`<|im_start|>assistant\n` 위치를 찾아서 **그 뒤(= 정답 글자 + 종료 토큰)만** 학습 대상으로 두고,
나머지는 전부 `-100`(무시)으로 마스킹합니다. 이게 정석입니다.

### 보기 순서 셔플 증강
매 에폭마다 보기 순서를 무작위로 섞고 정답 글자도 따라 바꿉니다.
→ **데이터가 실질적으로 4배**가 되고, "정답은 주로 a" 같은 편향 학습을 막습니다.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass, field
from typing import Any

def derive_assistant_header(processor):
    '''
    "<|im_start|>assistant\n" 같은 assistant 시작 표식을 chat template에서 직접 뽑아낸다.
    모델을 갈아끼웠을 때 표식이 달라도 자동으로 따라갑니다. (하드코딩 금지)
    '''
    probe = [{"role": "user", "content": [{"type": "text", "text": "Q"}]}]
    closed = processor.apply_chat_template(probe, tokenize=False, add_generation_prompt=False)
    opened = processor.apply_chat_template(probe, tokenize=False, add_generation_prompt=True)
    if opened.startswith(closed) and len(opened) > len(closed):
        tail = opened[len(closed):]
    else:
        tail = "<|im_start|>assistant\n"          # 폴백 (ChatML 계열)
    return tail, processor.tokenizer.encode(tail, add_special_tokens=False)

ASSIST_HEADER_TEXT, ASSIST_HEADER_IDS = derive_assistant_header(processor)
print("assistant 헤더:", repr(ASSIST_HEADER_TEXT), "->", ASSIST_HEADER_IDS)
assert ASSIST_HEADER_IDS, "assistant 헤더를 찾지 못했습니다 — chat template을 확인하세요."


def build_labels(input_ids, attention_mask):
    '''assistant 응답 부분(정답 글자)만 학습 대상으로 남기고 나머지는 -100'''
    labels = torch.full_like(input_ids, -100)
    hdr = torch.tensor(ASSIST_HEADER_IDS, dtype=input_ids.dtype)
    H = len(ASSIST_HEADER_IDS)
    for i in range(input_ids.size(0)):
        ids = input_ids[i]
        start = -1
        for j in range(ids.size(0) - H, -1, -1):          # 마지막 assistant 헤더를 뒤에서부터 탐색
            if torch.equal(ids[j:j + H], hdr):
                start = j + H
                break
        if start >= 0:
            labels[i, start:] = ids[start:]
    labels[attention_mask == 0] = -100                    # 패딩 제외
    return labels


class VQATrainDataset(Dataset):
    def __init__(self, df, shuffle_choices=True, seed=SEED):
        self.df = df.reset_index(drop=True)
        self.shuffle_choices = shuffle_choices
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, e):
        self.epoch = e

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        options = [row[c] for c in LETTERS]
        gold = LETTERS.index(row["answer"])

        if self.shuffle_choices:
            rng = random.Random((self.seed * 1_000_003) + (self.epoch * 7919) + i)
            perm = list(range(4)); rng.shuffle(perm)      # perm[k] = k번 자리에 놓을 원래 보기
            options = [options[perm[k]] for k in range(4)]
            gold = perm.index(gold)

        return {
            "image": load_image(row["abs_path"]),
            "messages": build_messages(row["question"], options, answer_letter=LETTERS[gold]),
        }


@dataclass
class TrainCollator:
    processor: Any
    def __call__(self, batch):
        tok = self.processor.tokenizer
        old = tok.padding_side
        tok.padding_side = "right"                        # 학습은 오른쪽 패딩
        try:
            texts = [self.processor.apply_chat_template(b["messages"], tokenize=False,
                                                        add_generation_prompt=False) for b in batch]
            enc = self.processor(text=texts, images=[b["image"] for b in batch],
                                 padding=True, return_tensors="pt")
        finally:
            tok.padding_side = old
        enc["labels"] = build_labels(enc["input_ids"], enc["attention_mask"])
        return enc


# ── 마스킹이 제대로 되는지 검사 (틀리면 여기서 즉시 중단) ────
set_pixels(processor, CFG["min_tokens"], CFG["train_max_tokens"])
_ds = VQATrainDataset(fit_df.head(2))
_b = TrainCollator(processor)([_ds[0], _ds[1]])
_n_sup = int((_b["labels"] != -100).sum(1)[0].item())
_sup = _b["labels"][0][_b["labels"][0] != -100]
_txt = tokenizer.decode(_sup)
print("입력 토큰 수:", tuple(_b["input_ids"].shape), "| 학습 대상 토큰 수:", _n_sup)
print("학습 대상 내용 →", repr(_txt))

# ★ 모델을 교체했을 때 헤더 탐색이 실패하면 전부 -100이 되어 "조용히" 학습이 무의미해집니다.
#   그걸 막기 위한 강제 검사입니다.
assert _n_sup > 0, (
    "라벨 마스킹 실패: 학습 대상 토큰이 0개입니다. "
    "assistant 헤더 탐색이 실패했을 가능성이 큽니다 (모델 교체 시 chat template 확인)."
)
assert _n_sup <= 8, f"학습 대상 토큰이 {_n_sup}개로 과다합니다 — 프롬프트까지 학습될 위험이 있습니다."
assert _txt.strip()[:1] in LETTERS, f"학습 대상이 정답 글자로 시작하지 않습니다: {_txt!r}"
print("✅ 정답 글자 + 종료토큰만 학습됩니다.")

---
# 9. LoRA 설정

- **비전 타워는 학습에서 제외**합니다. (베이스라인은 `gate_proj` 등 이름이 겹쳐서 비전 MLP까지
  의도치 않게 학습시킵니다 → 학습 파라미터가 늘고 과적합 위험만 커집니다.)
- `r=16, alpha=32, dropout=0.1` — 데이터가 수천 건이므로 r=8보다 조금 키우되 dropout으로 눌러줍니다.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import copy

SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

def language_lora_targets(model):
    '''언어모델 쪽 Linear 모듈의 "전체 이름"만 수집 (비전 타워 제외)'''
    names = []
    for n, m in model.named_modules():
        if m.__class__.__name__ not in ("Linear", "Linear4bit", "Linear8bitLt"):
            continue
        if "visual" in n or "vision" in n:
            continue
        if n.split(".")[-1] in SUFFIXES:
            names.append(n)
    return names

model = base_model
if CFG["run_finetune"]:
    targets = language_lora_targets(base_model)
    print(f"LoRA 적용 모듈 {len(targets)}개 (비전 타워 제외). 예: {targets[:3]}")

    if CFG["load_4bit"]:
        base_model = prepare_model_for_kbit_training(base_model,
                                                     use_gradient_checkpointing=True)
    else:
        base_model.enable_input_require_grads()
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.config.use_cache = False

    lora_cfg = LoraConfig(
        r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
        bias="none", target_modules=targets, task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
else:
    print("파인튜닝 건너뜀 (CFG['run_finetune'] = False)")

---
# 10. 학습 루프

### 과적합을 막는 3중 장치
1. **val *정확도*로 best 체크포인트 선택** — loss가 아니라 실제 점수 기준입니다.
   loss는 좋아지는데 정확도는 나빠지는 구간이 실제로 존재합니다.
2. **얼리스토핑** — 연속 2회 개선이 없으면 즉시 중단.
3. **마지막 안전장치** — 학습이 끝나도 "제로샷보다 나은가?"를 검증세트로 다시 확인하고,
   나쁘면 **파인튜닝 결과를 버립니다** (12번 셀).

In [ ]:
import math
from transformers import get_cosine_schedule_with_warmup

history = {"steps": [], "val_acc": [], "train_loss": []}
best_state, best_acc, best_step = None, -1.0, -1

if CFG["run_finetune"]:
    # 학습 데이터 구성 = train 분할 + dev 의사라벨
    parts = [fit_df]
    if pseudo_df is not None and len(pseudo_df):
        parts.append(pseudo_df[fit_df.columns.intersection(pseudo_df.columns)])
    full_fit = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    if CFG["max_train"]:
        full_fit = full_fit.head(CFG["max_train"]).reset_index(drop=True)
    print(f"실제 학습 데이터: {len(full_fit):,}건 "
          f"(train {len(fit_df):,} + dev의사라벨 {0 if pseudo_df is None else len(pseudo_df):,}, 상한 {CFG['max_train']})")

    train_ds = VQATrainDataset(full_fit, shuffle_choices=True)
    collator = TrainCollator(processor)

    steps_per_epoch = math.ceil(len(train_ds) / (CFG["train_bs"] * CFG["grad_accum"]))
    total_steps = steps_per_epoch * CFG["epochs"]
    eval_every = max(1, steps_per_epoch // CFG["evals_per_epoch"])

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * CFG["warmup_ratio"]), total_steps)
    use_scaler = (DTYPE == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    print(f"옵티마이저 스텝 {total_steps}회  (에폭당 {steps_per_epoch}, 검증 주기 {eval_every}스텝)")

    def run_eval(tag):
        set_pixels(processor, CFG["min_tokens"], CFG["infer_max_tokens"])
        p = predict_probs(model, val_quick_df, n_perm=CFG["train_n_perm"], desc=f"검증({tag})")
        set_pixels(processor, CFG["min_tokens"], CFG["train_max_tokens"])
        return accuracy(p, val_quick_df["answer"].tolist())

    # 학습 시작 전 기준선 (LoRA는 초기값이 항등이므로 사실상 제로샷)
    base_quick = run_eval("step0")
    print(f"[step 0] 검증 정확도(빠른) {base_quick*100:.2f}%  ← 넘어야 할 기준선")
    best_acc = base_quick
    history["steps"].append(0); history["val_acc"].append(base_quick); history["train_loss"].append(float("nan"))

    gstep, bad_rounds, stop = 0, 0, False
    for epoch in range(CFG["epochs"]):
        train_ds.set_epoch(epoch)
        loader = DataLoader(train_ds, batch_size=CFG["train_bs"], shuffle=True,
                            collate_fn=collator, num_workers=CFG["num_workers"],
                            pin_memory=False, drop_last=False)
        model.train(); model.config.use_cache = False
        optimizer.zero_grad(set_to_none=True)
        run_loss, seen = 0.0, 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}", unit="batch")
        for micro, batch in enumerate(pbar, start=1):
            batch = _to_device(batch, model)
            with torch.autocast("cuda", dtype=DTYPE, enabled=torch.cuda.is_available()):
                loss = model(**batch).loss / CFG["grad_accum"]
            scaler.scale(loss).backward() if use_scaler else loss.backward()
            run_loss += loss.item() * CFG["grad_accum"]; seen += 1
            del batch, loss

            if micro % CFG["grad_accum"] == 0 or micro == len(loader):
                if use_scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, CFG["max_grad_norm"])
                prev = scaler.get_scale() if use_scaler else None
                if use_scaler:
                    scaler.step(optimizer); scaler.update()
                    if scaler.get_scale() >= prev:
                        scheduler.step()
                else:
                    optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                gstep += 1
                pbar.set_postfix(loss=f"{run_loss/max(1,seen):.4f}", step=gstep, best=f"{best_acc*100:.1f}%")

                if gstep % eval_every == 0 or gstep == total_steps:
                    acc = run_eval(f"step{gstep}")
                    history["steps"].append(gstep); history["val_acc"].append(acc)
                    history["train_loss"].append(run_loss / max(1, seen))
                    flag = ""
                    if acc > best_acc:
                        best_acc, best_step, bad_rounds = acc, gstep, 0
                        best_state = {k: v.detach().cpu().clone()
                                      for k, v in get_peft_model_state_dict(model).items()}
                        flag = "  ⭐ best 갱신"
                    else:
                        bad_rounds += 1
                        flag = f"  (개선 없음 {bad_rounds}/{CFG['patience']})"
                    print(f"[step {gstep}] train_loss {run_loss/max(1,seen):.4f} | 검증 {acc*100:.2f}%{flag}")
                    run_loss, seen = 0.0, 0
                    model.train()
                    if bad_rounds >= CFG["patience"]:
                        print("⏹️ 조기 종료: 검증 정확도가 더 오르지 않습니다 (과적합 방지).")
                        stop = True; break
        if stop:
            break

    if best_state is not None:
        set_peft_model_state_dict(model, {k: v.to(model.device) for k, v in best_state.items()})
        print(f"\n✅ best 체크포인트 복원 (step {best_step}, 빠른검증 {best_acc*100:.2f}%)")
        os.makedirs(os.path.join(OUTPUT_DIR, "lora_best"), exist_ok=True)
        model.save_pretrained(os.path.join(OUTPUT_DIR, "lora_best"))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, "lora_best"))
        print("   저장:", os.path.join(OUTPUT_DIR, "lora_best"))
    else:
        print("\n⚠️ 학습 중 기준선을 한 번도 넘지 못했습니다 → 제로샷을 쓰게 됩니다.")

In [ ]:
# 학습 곡선
if CFG["run_finetune"] and len(history["steps"]) > 1:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3.6))
    # 라벨은 영문 — Colab 기본 환경에는 한글 폰트가 없어 □□□로 깨집니다.
    ax.plot(history["steps"], [a*100 for a in history["val_acc"]], "o-", label="val accuracy (%)")
    ax.axhline(history["val_acc"][0]*100, ls="--", c="gray", label="zero-shot baseline")
    ax.set_xlabel("optimizer step"); ax.set_ylabel("accuracy (%)")
    ax.set_title("Validation accuracy"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

---
# 11. ★ 최종 모델 자동 선택 (과적합 방어의 마지막 관문)

검증 세트에서 **세 가지 후보를 공정하게 겨루게** 하고 이긴 쪽으로 제출합니다.

| 후보 | 설명 |
|---|---|
| `zeroshot` | 파인튜닝 없이 사전학습 모델 그대로 |
| `finetuned` | LoRA best 체크포인트 |
| `ensemble` | 두 확률을 평균 |

> **동점·박빙이면 단순한 쪽(zeroshot)을 고릅니다.**
> 검증 400문항에서 ±2%p 정도는 그냥 운입니다. 그래서 `+0.5%p` 이상 확실히 이길 때만
> 복잡한 후보로 갈아탑니다. 이것이 **검증 세트에 과적합되는 것**까지 막아줍니다.

### 정답 분포 사전확률 보정
학습 데이터의 정답이 한쪽으로 쏠려 있으면(예: a가 40%) 그 정보를 약하게 반영할 수 있습니다.
`최종점수 = log(모델확률) + α · log(정답분포)` 에서 **α를 검증세트로 고릅니다.**
효과가 없으면 자동으로 `α=0`(미적용)이 선택되므로 **손해 볼 일이 없습니다.**

In [ ]:
LOG_PRIOR = np.log(np.clip(ANSWER_PRIOR, 1e-6, None))
ALPHA_GRID = [0.0, 0.15, 0.3, 0.5, 0.75, 1.0]

def apply_prior(probs, alpha):
    if alpha == 0.0:
        return probs
    s = np.log(np.clip(probs, 1e-9, None)) + alpha * LOG_PRIOR
    s = s - s.max(axis=1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=1, keepdims=True)

def tune_alpha(probs, gold):
    best_a, best_acc = 0.0, accuracy(probs, gold)
    for a in ALPHA_GRID[1:]:
        acc = accuracy(apply_prior(probs, a), gold)
        if acc > best_acc + 1e-9:
            best_a, best_acc = a, acc
    return best_a, best_acc


gold = val_eval_df["answer"].tolist()
candidates = {"zeroshot": zs_val_probs}

if CFG["run_finetune"] and best_state is not None:
    ft_val_probs = predict_probs(model, val_eval_df, desc="파인튜닝 검증")
    candidates["finetuned"] = ft_val_probs
    candidates["ensemble"] = (zs_val_probs + ft_val_probs) / 2.0

print("\n" + "=" * 62)
print(f"{'후보':<12}{'α=0 정확도':>14}{'최적 α':>10}{'보정 후':>14}")
print("-" * 62)
results = {}
for name, p in candidates.items():
    a, acc = tune_alpha(p, gold)
    results[name] = (a, acc, accuracy(p, gold))
    print(f"{name:<12}{accuracy(p, gold)*100:>13.2f}%{a:>10.2f}{acc*100:>13.2f}%")
print("=" * 62)

# 단순한 후보를 기본으로 두고, 0.5%p 이상 확실히 이길 때만 교체
ORDER = ["zeroshot", "finetuned", "ensemble"]
MARGIN = 0.005
best_name = "zeroshot"
for name in ORDER[1:]:
    if name in results and results[name][1] > results[best_name][1] + MARGIN:
        best_name = name
BEST_ALPHA, BEST_VAL_ACC, _ = results[best_name]

print(f"\n🏆 최종 선택: {best_name}  (α={BEST_ALPHA}, 검증 정확도 {BEST_VAL_ACC*100:.2f}%)")
print(f"   제로샷 대비 {(BEST_VAL_ACC - results['zeroshot'][1])*100:+.2f}%p")
if best_name == "zeroshot" and "finetuned" in results:
    print("   → 파인튜닝이 의미 있는 개선을 주지 못했습니다. 과적합을 피하려 제로샷을 씁니다.")

---
# 12. 테스트 추론 & 제출 파일 생성

여기서 시간이 가장 오래 걸립니다. 진행바의 남은 시간을 확인하세요.
`ensemble`이 선택된 경우 테스트를 두 번 돌기 때문에 시간이 2배입니다.

In [ ]:
import time
t0 = time.time()

if best_name == "zeroshot":
    test_probs = predict_probs(model, test_df, desc="테스트(제로샷)", disable_adapter=True)
elif best_name == "finetuned":
    test_probs = predict_probs(model, test_df, desc="테스트(파인튜닝)")
else:
    p_ft = predict_probs(model, test_df, desc="테스트(파인튜닝) 1/2")
    p_zs = predict_probs(model, test_df, desc="테스트(제로샷) 2/2", disable_adapter=True)
    test_probs = (p_ft + p_zs) / 2.0

test_probs = apply_prior(test_probs, BEST_ALPHA)
preds = probs_to_letters(test_probs)
print(f"\n추론 완료 ({(time.time()-t0)/60:.1f}분)")

In [ ]:
# ── 제출 파일 생성 ──────────────────────────────────────────
submission = pd.DataFrame({"id": test_df["id"].values, "answer": preds})

# sample_submission이 있으면 id 순서/구성을 그대로 맞춘다
if sample_sub is not None and "id" in sample_sub.columns:
    merged = sample_sub[["id"]].merge(submission, on="id", how="left")
    n_missing = merged["answer"].isna().sum()
    if n_missing:
        print(f"⚠️ sample_submission에는 있으나 예측이 없는 id {n_missing}건 → 'a'로 채움")
        merged["answer"] = merged["answer"].fillna("a")
    submission = merged

for path in [os.path.join(OUTPUT_DIR, "submission.csv"), "submission.csv"]:
    submission.to_csv(path, index=False)

# ── 검증 ────────────────────────────────────────────────────
assert list(submission.columns) == ["id", "answer"], submission.columns
assert submission["answer"].isin(LETTERS).all(), "a~d 이외의 값이 있습니다"
assert submission["id"].is_unique, "중복 id가 있습니다"
assert len(submission) == (len(sample_sub) if sample_sub is not None else len(test_df))

print("✅ submission.csv 저장 완료 —", len(submission), "행")
print("   ", os.path.join(OUTPUT_DIR, "submission.csv"))
print("\n예측 분포:")
vc = submission["answer"].value_counts(normalize=True).reindex(LETTERS).fillna(0)
for L in LETTERS:
    print(f"  {L}: {vc[L]*100:5.1f} %  {'█'*int(vc[L]*100)}")
print("\n상위 5행:")
print(submission.head().to_string(index=False))

# 확신도 분포 (낮은 문항이 많으면 해상도를 더 올려볼 여지가 있습니다)
conf = test_probs.max(axis=1)
print(f"\n평균 확신도 {conf.mean():.3f} | 0.5 미만 문항 {int((conf<0.5).sum())}건 "
      f"({(conf<0.5).mean()*100:.1f}%)")

In [ ]:
# ── 실험 요약 리포트 저장 (제출용 '추가 문서'에 붙여 쓰세요) ──
import json, datetime

report = {
    "생성시각": datetime.datetime.now().isoformat(timespec="seconds"),
    "모델": MODEL_ID,
    "모델키": MODEL_KEY,
    "프리셋": PRESET,
    "4bit양자화": CFG["load_4bit"],
    "토큰당픽셀": f"{PX_UNIT}x{PX_UNIT}",
    "학습토큰예산": CFG["train_max_tokens"],
    "추론토큰예산": CFG["infer_max_tokens"],
    "추론실효해상도": f"~{int(tokens_to_px(CFG['infer_max_tokens'])**0.5)}px",
    "TTA_보기순환치환": CFG["n_perm"],
    "학습데이터수": int(len(fit_df)) if CFG["run_finetune"] else 0,
    "dev의사라벨수": 0 if pseudo_df is None else int(len(pseudo_df)),
    "검증세트크기": int(len(val_eval_df)),
    "검증정확도": {k: round(v[1] * 100, 2) for k, v in results.items()},
    "최종선택": best_name,
    "사전확률보정_alpha": BEST_ALPHA,
    "최종검증정확도": round(BEST_VAL_ACC * 100, 2),
    "학습곡선": {"steps": history["steps"], "val_acc": [round(a*100, 2) for a in history["val_acc"]]},
}
with open(os.path.join(OUTPUT_DIR, "report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))

---
# 13. 더 올리고 싶다면 (우선순위 순)

1. **해상도를 더 올린다** — `CFG['infer_max_tokens']`를 `1280 → 1600 → 2048`.
   OOM만 안 나면 거의 항상 이득입니다. **가장 가성비 좋은 손잡이**입니다.
2. **더 큰 GPU에서 7B를 4bit 없이** 돌린다 (`load_4bit=False`).
   양자화는 정확도를 조금 깎습니다.
3. **`n_perm`은 4를 유지**하세요. 1로 줄이면 보통 1~3%p 손해입니다.
4. **에폭을 2로** 늘려보고 학습 곡선을 확인하세요.
   검증 정확도가 계속 오르면 데이터가 더 필요한 것이고, 꺾이면 이미 충분합니다.
5. **`dev_min_agree`를 3으로** 낮춰 의사라벨을 늘려보세요.
   단, 검증 정확도가 떨어지면 되돌리세요.
6. **다중 해상도 앙상블** — 같은 모델을 `infer_max_tokens` 1024와 1600으로 두 번 돌려
   확률을 평균내면 작은 글씨/큰 간판을 모두 커버합니다. (시간 2배)

### 하지 말아야 할 것
- ❌ 검증 정확도를 보고 하이퍼파라미터를 수십 번 바꾸기 → **검증 세트에 과적합**됩니다.
  이 노트북은 후보를 3개로 제한하고 `+0.5%p` 마진을 둬서 그걸 막고 있습니다.
- ❌ 에폭을 5~10으로 늘리기 → 4지선다 한 글자 예측은 금방 외워버립니다.
- ❌ `test` 이미지를 학습에 쓰기 → 규정 위반입니다.